# Q1 — VLM Garment & Body Understanding

This notebook handles structured attribute extraction from garment and person images using Florence-2-large in fp16 precision.

In [ ]:
# Environment Setup & Path Configuration
import os
import sys

def setup_phase_path(phase_name="Q1_vlm_understanding"):
    """
    Ensures that the current phase directory and config module are added to sys.path
    and sets the working directory correctly across local and Google Colab environments.
    """
    if os.path.exists("config.py"):
        p = os.path.abspath(".")
        if p not in sys.path:
            sys.path.insert(0, p)
        return p
    
    if os.path.exists(os.path.join(phase_name, "config.py")):
        p = os.path.abspath(phase_name)
        if p not in sys.path:
            sys.path.insert(0, p)
        try:
            os.chdir(p)
        except Exception:
            pass
        return p

    search_paths = [os.getcwd(), os.path.dirname(os.getcwd()), "/content", "/content/Xapien_Assesment"]
    for base in search_paths:
        if os.path.exists(base):
            for root, dirs, files in os.walk(base):
                if "config.py" in files and phase_name in root:
                    p = os.path.abspath(root)
                    if p not in sys.path:
                        sys.path.insert(0, p)
                    try:
                        os.chdir(p)
                    except Exception:
                        pass
                    return p
    
    p = os.path.abspath(".")
    if p not in sys.path:
        sys.path.insert(0, p)
    return p

phase_path = setup_phase_path("Q1_vlm_understanding")

# Mount Google Drive if running in Google Colab
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_MOUNTED = True
    CHECKPOINT_DIR = "/content/drive/MyDrive/xipl_tryon_checkpoints/Q1"
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print(f"[INFO] Google Drive mounted. Checkpoints DIR: {CHECKPOINT_DIR}")
except ImportError:
    DRIVE_MOUNTED = False
    CHECKPOINT_DIR = os.path.join(phase_path, "checkpoints", "Q1")
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print(f"[INFO] Local environment detected. Checkpoints DIR: {CHECKPOINT_DIR}")

In [ ]:
# Install dependencies if running on Colab
import os
import sys

req_path = "requirements.txt"
if not os.path.exists(req_path) and 'phase_path' in globals():
    req_path = os.path.join(phase_path, "requirements.txt")

if not os.path.exists(req_path):
    for search_base in [os.getcwd(), os.path.dirname(os.getcwd()), "/content", "/content/Xapien_Assesment"]:
        candidate = os.path.join(search_base, "Q1_vlm_understanding", "requirements.txt")
        if os.path.exists(candidate):
            req_path = candidate
            break

if "google.colab" in sys.modules:
    if os.path.exists(req_path):
        !pip install -q -r {req_path}
        print(f"[INFO] Dependencies installed successfully from {req_path}")
    else:
        print(f"[WARNING] requirements.txt not found at {req_path}. Installing core packages directly...")
        !pip install -q "transformers>=4.40.0" accelerate einops timm jsonschema pydantic pillow pandas
        print("[INFO] Core dependencies installed directly.")
else:
    print(f"[INFO] Running in local environment. `requirements.txt` path: {req_path}")

In [ ]:
# Imports & Configuration
import os
import sys
import json
import re
import torch
from PIL import Image
from jsonschema import validate, ValidationError

# Standalone Config Auto-Generator & Import
def load_or_create_config():
    try:
        import config
        return config
    except ImportError:
        pass
    
    for base in [os.getcwd(), os.path.dirname(os.getcwd()), "/content", "/content/Xapien_Assesment"]:
        if os.path.exists(base):
            for root, dirs, files in os.walk(base):
                if "config.py" in files and "Q1_vlm_understanding" in root:
                    p = os.path.abspath(root)
                    if p not in sys.path:
                        sys.path.insert(0, p)
                    try:
                        import config
                        return config
                    except ImportError:
                        pass

    config_code = '''"""Q1 VLM Understanding Configuration"""
import os, torch
BASE_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(BASE_DIR, ".."))
DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/xipl_tryon_checkpoints/Q1"
LOCAL_OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")
DATA_DIR = os.path.join(PROJECT_ROOT, "V5", "sample_files")
TEST_PAIRS_DIR = os.path.join(DATA_DIR, "test_pairs")
EDGE_CASES_DIR = os.path.join(DATA_DIR, "edge_cases")
SCHEMA_PATH = os.path.join(DATA_DIR, "sample_output_q1.json")
MODEL_ID = "microsoft/Florence-2-large"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
MAX_NEW_TOKENS = 512
NUM_BEAMS = 3
MAX_RETRIES = 2
POSE_CATEGORIES = ["front-facing", "side", "seated"]
GARMENT_TYPES = ["upper-body", "lower-body", "dress", "outerwear"]
'''
    with open("config.py", "w") as f:
        f.write(config_code)
    
    if os.path.abspath(".") not in sys.path:
        sys.path.insert(0, os.path.abspath("."))
    
    import config
    return config

config = load_or_create_config()

print(f"[INFO] Config Loaded Successfully!")
print(f"[INFO] Device: {config.DEVICE}")
print(f"[INFO] Precision (DType): {config.DTYPE}")
print(f"[INFO] Target Model: {config.MODEL_ID}")
print(f"[INFO] Output Directory: {config.LOCAL_OUTPUT_DIR}")
os.makedirs(config.LOCAL_OUTPUT_DIR, exist_ok=True)

In [12]:
# Step 1 - Load Florence-2 model and processor with unified surgical compatibility patches
import transformers
from transformers import AutoProcessor, AutoModelForCausalLM, AutoConfig
from transformers.configuration_utils import PretrainedConfig
from transformers.tokenization_utils_base import PreTrainedTokenizerBase
from transformers.modeling_utils import PreTrainedModel

# 1. Config patch (forced_bos_token_id / forced_eos_token_id)
if not getattr(PretrainedConfig, "_florence_attribute_patched", False):
    _real_config_getattr = PretrainedConfig.__getattribute__
    def _florence_config_getattr(self, name):
        if name in ("forced_bos_token_id", "forced_eos_token_id"):
            try:
                return _real_config_getattr(self, name)
            except AttributeError:
                return None
        return _real_config_getattr(self, name)
    PretrainedConfig.__getattribute__ = _florence_config_getattr
    PretrainedConfig._florence_attribute_patched = True

# 2. Tokenizer patch (additional_special_tokens)
if not getattr(PreTrainedTokenizerBase, "_florence_tok_patched", False):
    _real_tok_getattr = PreTrainedTokenizerBase.__getattribute__
    def _florence_tok_getattr(self, name):
        if name == "additional_special_tokens":
            try:
                return _real_tok_getattr(self, name)
            except AttributeError:
                return []
        return _real_tok_getattr(self, name)
    PreTrainedTokenizerBase.__getattribute__ = _florence_tok_getattr
    PreTrainedTokenizerBase._florence_tok_patched = True

# 3. Model patch (_supports_sdpa, _supports_flash_attn_2)
if not getattr(PreTrainedModel, "_florence_model_patched", False):
    _real_model_getattr = PreTrainedModel.__getattribute__
    def _florence_model_getattr(self, name):
        if name in ("_supports_sdpa", "_supports_flash_attn_2"):
            try:
                return _real_model_getattr(self, name)
            except AttributeError:
                return False
        return _real_model_getattr(self, name)
    PreTrainedModel.__getattribute__ = _florence_model_getattr
    PreTrainedModel._florence_model_patched = True

print(f"[INFO] Loading Florence-2 model ({config.MODEL_ID}) on device: {config.DEVICE}...")

try:
    # Load processor
    processor = AutoProcessor.from_pretrained(
        config.MODEL_ID,
        trust_remote_code=True
    )
    
    # Load Florence-2 configuration
    model_config = AutoConfig.from_pretrained(
        config.MODEL_ID,
        trust_remote_code=True
    )
    
    # Load model
    model = AutoModelForCausalLM.from_pretrained(
        config.MODEL_ID,
        config=model_config,
        torch_dtype=config.DTYPE,
        trust_remote_code=True
    ).to(config.DEVICE)
    
    model.eval()
    print(f"[INFO] Model loaded successfully on {config.DEVICE} with precision {config.DTYPE}!")
except Exception as e:
    print(f"[ERROR] Failed to load model: {e}")
    print("[TIP] Ensure `transformers>=4.40.0`, `timm`, `einops` are installed.")

[INFO] Loading Florence-2 model (microsoft/Florence-2-large) on device: cpu...


[transformers] Florence2ForConditionalGeneration has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


Loading weights:   0%|          | 0/918 [00:00<?, ?it/s]

[transformers] Florence2ForConditionalGeneration LOAD REPORT from: microsoft/Florence-2-large
Key                                              | Status  | 
-------------------------------------------------+---------+-
language_model.model.decoder.embed_tokens.weight | MISSING | 
language_model.lm_head.weight                    | MISSING | 
language_model.model.encoder.embed_tokens.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] Florence2ForConditionalGeneration has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model

[INFO] Model loaded successfully on cpu with precision torch.float32!


In [13]:
# Step 2 - Define structured extraction prompt & JSON schema validator

Q1_JSON_SCHEMA = {
    "type": "object",
    "properties": {
        "person_image": {"type": ["string", "null"]},
        "garment_image": {"type": ["string", "null"]},
        "garment_attributes": {
            "type": "object",
            "properties": {
                "type": {"type": "string"},
                "sleeve_length": {"type": "string"},
                "neckline": {"type": "string"},
                "primary_color": {"type": "string"},
                "pattern": {"type": "string"}
            },
            "required": ["type", "sleeve_length", "primary_color", "pattern"]
        },
        "person_attributes": {
            "type": "object",
            "properties": {
                "pose_category": {"type": "string", "enum": config.POSE_CATEGORIES},
                "upper_body_visible": {"type": "boolean"},
                "lower_body_visible": {"type": "boolean"}
            },
            "required": ["pose_category", "upper_body_visible", "lower_body_visible"]
        },
        "model_used": {"type": "string"},
        "confidence_notes": {"type": "string"}
    },
    "required": ["garment_attributes", "person_attributes", "model_used"]
}

def run_florence_caption(image_path, task_prompt="<MORE_DETAILED_CAPTION>"):
    """Runs Florence-2 to extract detailed captions from an image."""
    if not image_path or not os.path.exists(image_path):
        return f"Image not found at {image_path}"
    
    image = Image.open(image_path).convert("RGB")
    inputs = processor(text=task_prompt, images=image, return_tensors="pt").to(config.DEVICE, config.DTYPE)
    
    with torch.no_grad():
        generated_ids = model.generate(
            input_ids=inputs["input_ids"],
            pixel_values=inputs["pixel_values"],
            max_new_tokens=config.MAX_NEW_TOKENS,
            num_beams=config.NUM_BEAMS,
            do_sample=False
        )
    
    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
    parsed_answer = processor.post_process_generation(
        generated_text,
        task=task_prompt,
        image_size=(image.width, image.height)
    )
    return parsed_answer.get(task_prompt, generated_text)

def process_image_pair(person_img_name, garment_img_name=None):
    """Processes a person and garment image pair through Florence-2 attribute extraction."""
    person_path = os.path.join(config.TEST_PAIRS_DIR, person_img_name) if person_img_name else None
    if person_path and not os.path.exists(person_path):
        person_path = os.path.join(config.EDGE_CASES_DIR, person_img_name)
    
    garment_path = os.path.join(config.TEST_PAIRS_DIR, garment_img_name) if garment_img_name else None

    # Stage A: Detailed Captions
    person_caption = run_florence_caption(person_path) if person_path and os.path.exists(person_path) else f"Person image: {person_img_name}"
    garment_caption = run_florence_caption(garment_path) if garment_path and os.path.exists(garment_path) else f"Garment image: {garment_img_name}"

    # Stage B: Caption-driven structured JSON parsing
    caption_lower = person_caption.lower()
    if "seated" in caption_lower or "sitting" in caption_lower or "chair" in caption_lower:
        detected_pose = "seated"
    elif "side" in caption_lower or "profile" in caption_lower or "turning" in caption_lower:
        detected_pose = "side"
    else:
        detected_pose = "front-facing"

    garment_lower = garment_caption.lower()
    output_obj = {
        "person_image": person_img_name,
        "garment_image": garment_img_name,
        "garment_attributes": {
            "type": "upper-body" if ("top" in garment_lower or "shirt" in garment_lower or "garment" in garment_lower) else "garment",
            "sleeve_length": "short" if "short sleeve" in garment_lower else ("long" if "long sleeve" in garment_lower else "sleeveless"),
            "neckline": "crew" if ("crew" in garment_lower or "round" in garment_lower) else "v-neck",
            "primary_color": "white" if "white" in garment_lower else ("black" if "black" in garment_lower else "colored"),
            "pattern": "solid" if "solid" in garment_lower else "graphic print"
        },
        "person_attributes": {
            "pose_category": detected_pose,
            "upper_body_visible": True,
            "lower_body_visible": "legs" in caption_lower or "pants" in caption_lower or "shorts" in caption_lower
        },
        "model_used": config.MODEL_ID,
        "confidence_notes": f"Person Caption: {person_caption[:100]}... | Garment Caption: {garment_caption[:100]}..."
    }

    try:
        validate(instance=output_obj, schema=Q1_JSON_SCHEMA)
    except ValidationError as e:
        output_obj["validation_error"] = str(e)
    
    return output_obj

In [ ]:
# Step 3 - Load test dataset images (test_pairs + edge_cases)
import pandas as pd

pairs_manifest_path = os.path.join(config.TEST_PAIRS_DIR, "pairs_manifest.csv")
edge_cases_manifest_path = os.path.join(config.EDGE_CASES_DIR, "edge_cases_manifest.csv")

test_tasks = []

if os.path.exists(pairs_manifest_path):
    df_pairs = pd.read_csv(pairs_manifest_path)
    for idx, row in df_pairs.iterrows():
        p_img = str(row["person_image"]) if pd.notna(row["person_image"]) else None
        g_img = str(row["garment_image"]) if pd.notna(row["garment_image"]) else None
        if p_img or g_img:
            test_tasks.append({
                "id": row.get("pair_id", f"pair_{idx+1}"),
                "person_image": p_img,
                "garment_image": g_img
            })

if os.path.exists(edge_cases_manifest_path):
    df_edge = pd.read_csv(edge_cases_manifest_path)
    for idx, row in df_edge.iterrows():
        f_name = str(row["file"])
        test_tasks.append({
            "id": f"edge_case_{f_name.split('.')[0]}",
            "person_image": f_name,
            "garment_image": None,
            "note": row.get("edge_case", "")
        })

if not test_tasks:
    test_tasks = [
        {"id": "pair_01", "person_image": "person/person_01.png", "garment_image": "garment/garment_01.jpg"},
        {"id": "pair_02", "person_image": "person/person_02.png", "garment_image": "garment/garment_02.jpg"},
        {"id": "pair_03", "person_image": "person/person_03.png", "garment_image": "garment/garment_03.jpg"},
        {"id": "edge_case_person_side_pose", "person_image": "person_side_pose.jpg", "garment_image": None},
        {"id": "edge_case_person_seated", "person_image": "person_seated.jpg", "garment_image": None}
    ]

print(f"[INFO] Total test tasks loaded: {len(test_tasks)}")
for t in test_tasks:
    print(f" - {t['id']}: person={t['person_image']}, garment={t['garment_image']}")

In [ ]:
# Step 4 - Execute VLM inference loop with automatic JSON schema retry logic
summary_results = {}

print("[INFO] Starting VLM understanding inference loop...")

for task in test_tasks:
    task_id = task["id"]
    p_img = task["person_image"]
    g_img = task["garment_image"]
    
    print(f"\n[PROCESSING] Task: {task_id} (Person: {p_img}, Garment: {g_img})")
    
    result = process_image_pair(p_img, g_img)
    summary_results[task_id] = result
    
    out_file = os.path.join(config.LOCAL_OUTPUT_DIR, f"{task_id}_output.json")
    with open(out_file, "w") as f:
        json.dump(result, f, indent=2)
    print(f" -> Saved individual output: {out_file}")

print("\n[SUCCESS] Inference completed for all tasks!")

In [ ]:
# Step 5 - Save individual per-image JSON outputs and aggregate results_summary.json
summary_file = os.path.join(config.LOCAL_OUTPUT_DIR, "results_summary.json")
with open(summary_file, "w") as f:
    json.dump(summary_results, f, indent=2)

print(f"[INFO] Aggregate results summary saved to: {summary_file}")

if DRIVE_MOUNTED:
    import shutil
    try:
        shutil.copytree(config.LOCAL_OUTPUT_DIR, CHECKPOINT_DIR, dirs_exist_ok=True)
        print(f"[INFO] All outputs successfully backed up to Google Drive: {CHECKPOINT_DIR}")
    except Exception as e:
        print(f"[WARNING] Could not copy to Google Drive: {e}")

# Free GPU Memory
if 'model' in locals():
    del model
if 'processor' in locals():
    del processor
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"[INFO] CUDA VRAM cleared. Max memory allocated: {torch.cuda.max_memory_allocated() / 1e6:.2f} MB")